In [1]:
import scanpy as sc
import glob
import os
import pandas as pd

from matplotlib import rcParams

In [2]:
pd.set_option('display.max_rows', 200)

In [3]:
data_path = "/mnt/nfs/EMBO/Integration/data/Hickey2023/all/"

In [4]:
# List folders with 10x files
file_paths = glob.glob(data_path + "B0*")
file_paths[0:10]

['/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B005-A-001',
 '/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B005-A-002',
 '/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B008-A-201',
 '/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B005-A-501',
 '/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B012-A-401',
 '/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B006-A-201',
 '/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B010-A-201',
 '/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B005-A-501-R2',
 '/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B012-A-405',
 '/mnt/nfs/EMBO/Integration/data/Hickey2023/all/B010-A-301']

In [5]:
len(file_paths)

75

In [6]:
# Load all data
adatas = [sc.read_10x_mtx(f) for f in file_paths]

In [7]:
# Parse objects with sample information
for i in range(len(file_paths)):
    # Get sample ID from path
    sample_id = os.path.basename(file_paths[i])
    
    # set sample
    adatas[i].obs['sample_id'] = sample_id

    # store original barcodes
    adatas[i].obs['barcode'] = adatas[i].obs.index

    # set cell index barcodes by preceeded by sample
    adatas[i].obs.index = sample_id + "_" + adatas[i].obs.index

In [8]:
adatas[0:5]

[AnnData object with n_obs × n_vars = 6794880 × 33694
     obs: 'sample_id', 'barcode'
     var: 'gene_ids', 'feature_types',
 AnnData object with n_obs × n_vars = 6794880 × 33694
     obs: 'sample_id', 'barcode'
     var: 'gene_ids', 'feature_types',
 AnnData object with n_obs × n_vars = 689803 × 36601
     obs: 'sample_id', 'barcode'
     var: 'gene_ids', 'feature_types',
 AnnData object with n_obs × n_vars = 6794880 × 33538
     obs: 'sample_id', 'barcode'
     var: 'gene_ids', 'feature_types',
 AnnData object with n_obs × n_vars = 722221 × 36601
     obs: 'sample_id', 'barcode'
     var: 'gene_ids', 'feature_types']

In [86]:
# Combine anndata objects
adata = sc.concat(adatas)

In [87]:
adata

AnnData object with n_obs × n_vars = 223988123 × 21812
    obs: 'sample_id', 'barcode'

In [88]:
adata.obs

,sample_id,barcode
B005-A-001_AAACCCAAGAAACACT-1,B005-A-001,AAACCCAAGAAACACT-1
B005-A-001_AAACCCAAGAAACCAT-1,B005-A-001,AAACCCAAGAAACCAT-1
B005-A-001_AAACCCAAGAAACCCA-1,B005-A-001,AAACCCAAGAAACCCA-1
B005-A-001_AAACCCAAGAAACCCG-1,B005-A-001,AAACCCAAGAAACCCG-1
B005-A-001_AAACCCAAGAAACCTG-1,B005-A-001,AAACCCAAGAAACCTG-1
...,...,...
B001-A-001_TTTGTTGTCTTTGCTA-1,B001-A-001,TTTGTTGTCTTTGCTA-1
B001-A-001_TTTGTTGTCTTTGCTG-1,B001-A-001,TTTGTTGTCTTTGCTG-1
B001-A-001_TTTGTTGTCTTTGGAG-1,B001-A-001,TTTGTTGTCTTTGGAG-1
B001-A-001_TTTGTTGTCTTTGGCT-1,B001-A-001,TTTGTTGTCTTTGGCT-1


## Load and merge metadata

In [89]:
# Load metadata
metadata = pd.read_csv(data_path + "sample_location_metadata.csv")
metadata.head()

,SampleNameRNA,SampleNameOnly,Donor,Multiome,Location
0,B001-A-001,B001-A-001,B001,No,Mid-jejunum
1,B001-A-006,B001-A-006,B001,No,Ileum
2,B001-A-101,B001-A-101,B001,No,Proximal-jejunum
3,B001-A-201,B001-A-201,B001,No,Duodenum
4,B001-A-301,B001-A-301,B001,No,Sigmoid


In [90]:
# rename columns
metadata = metadata.rename(columns={"SampleNameRNA": "sample_id"})

In [91]:
adata.obs

,sample_id,barcode
B005-A-001_AAACCCAAGAAACACT-1,B005-A-001,AAACCCAAGAAACACT-1
B005-A-001_AAACCCAAGAAACCAT-1,B005-A-001,AAACCCAAGAAACCAT-1
B005-A-001_AAACCCAAGAAACCCA-1,B005-A-001,AAACCCAAGAAACCCA-1
B005-A-001_AAACCCAAGAAACCCG-1,B005-A-001,AAACCCAAGAAACCCG-1
B005-A-001_AAACCCAAGAAACCTG-1,B005-A-001,AAACCCAAGAAACCTG-1
...,...,...
B001-A-001_TTTGTTGTCTTTGCTA-1,B001-A-001,TTTGTTGTCTTTGCTA-1
B001-A-001_TTTGTTGTCTTTGCTG-1,B001-A-001,TTTGTTGTCTTTGCTG-1
B001-A-001_TTTGTTGTCTTTGGAG-1,B001-A-001,TTTGTTGTCTTTGGAG-1
B001-A-001_TTTGTTGTCTTTGGCT-1,B001-A-001,TTTGTTGTCTTTGGCT-1


In [92]:
adata.obs = pd.merge(adata.obs, metadata, on="sample_id")

/mnt/nfs/EMBO/miniforge3/envs/scanpy2/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [93]:
adata.obs.head()

,sample_id,barcode,SampleNameOnly,Donor,Multiome,Location
0,B005-A-001,AAACCCAAGAAACACT-1,B005-A-001,B005,No,Sigmoid
1,B005-A-001,AAACCCAAGAAACCAT-1,B005-A-001,B005,No,Sigmoid
2,B005-A-001,AAACCCAAGAAACCCA-1,B005-A-001,B005,No,Sigmoid
3,B005-A-001,AAACCCAAGAAACCCG-1,B005-A-001,B005,No,Sigmoid
4,B005-A-001,AAACCCAAGAAACCTG-1,B005-A-001,B005,No,Sigmoid


## Basic filtering of cells and genes
Reduces file size

In [94]:
sc.pp.filter_cells(adata, min_counts=100)

In [95]:
sc.pp.filter_genes(adata, min_cells=3)

In [96]:
adata

AnnData object with n_obs × n_vars = 4141747 × 21332
    obs: 'sample_id', 'barcode', 'SampleNameOnly', 'Donor', 'Multiome', 'Location', 'n_counts'
    var: 'n_cells'

In [97]:
adata.obs.sample_id.value_counts()

sample_id
B005-A-002       96189
B005-A-401       94123
B005-A-301       89521
B009-A-401       89019
B001-A-301       88949
B001-A-101       88869
B004-A-104       88623
B008-A-401       85247
B004-A-204       84845
B001-A-401       84296
B005-A-501-R2    84112
B008-A-501       83650
B005-A-501       83324
B001-A-501       83306
B012-A-501       82236
B004-A-404       80889
B001-A-201       80869
B004-A-104-R2    80490
B001-A-006       78937
B004-A-408-R2    78750
B004-A-304       78495
B012-A-201       77514
B004-A-504       77259
B001-A-001       75544
B012-A-301       74157
B008-A-301       74072
B012-A-401       72763
B010-A-501       72740
B004-A-408       72456
B012-A-101       72447
B010-A-405       72273
B011-A-405       72260
B010-A-101       72183
B011-A-401       71847
B004-A-004       71569
B010-A-201       70978
B011-A-002       70793
B010-A-301       69386
B010-A-401       67695
B005-A-402       67575
B011-A-001       66174
B004-A-404-R2    64719
B011-A-301       61900
B

In [99]:
adata.obs

,sample_id,barcode,SampleNameOnly,Donor,Multiome,Location,n_counts
1461,B005-A-001,AAACCCAAGTAATACG-1,B005-A-001,B005,No,Sigmoid,117.0
6207,B005-A-001,AAACCCATCGAGTGAG-1,B005-A-001,B005,No,Sigmoid,120.0
10734,B005-A-001,AAACGAAGTACCCGAC-1,B005-A-001,B005,No,Sigmoid,135.0
10892,B005-A-001,AAACGAAGTAGGTCAG-1,B005-A-001,B005,No,Sigmoid,3878.0
13473,B005-A-001,AAACGAATCGGCACTG-1,B005-A-001,B005,No,Sigmoid,10599.0
...,...,...,...,...,...,...,...
223987948,B001-A-001,TTTGTTGTCTGCTCTG-1,B001-A-001,B001,No,Mid-jejunum,639.0
223987958,B001-A-001,TTTGTTGTCTGGACTA-1,B001-A-001,B001,No,Mid-jejunum,542.0
223987996,B001-A-001,TTTGTTGTCTGTCCCA-1,B001-A-001,B001,No,Mid-jejunum,582.0
223988065,B001-A-001,TTTGTTGTCTTCTGGC-1,B001-A-001,B001,No,Mid-jejunum,588.0


In [83]:
adata.write_h5ad("/mnt/nfs/EMBO/Integration/data/Hickey2023/hickey.h5ad")

In [100]:
adata

AnnData object with n_obs × n_vars = 4141747 × 21332
    obs: 'sample_id', 'barcode', 'SampleNameOnly', 'Donor', 'Multiome', 'Location', 'n_counts'
    var: 'n_cells'

## Subsample data

In [101]:
# # subset data
# adata_sub = sc.pp.subsample(adata, fraction=0.05, copy=True)

In [102]:
# adata_sub

AnnData object with n_obs × n_vars = 207087 × 21332
    obs: 'sample_id', 'barcode', 'SampleNameOnly', 'Donor', 'Multiome', 'Location', 'n_counts'
    var: 'n_cells'

In [106]:
# adata_sub.write_h5ad("/mnt/nfs/EMBO/Integration/data/Hickey2023/hickey_subsample.h5ad")